# Exp 022 — Persona prompt + word-ban (Blind-A ship)

**Challenger to exp 021 (rank 9/9, composite 0.33).**

Single-axis change: stock response prompt → persona+word-ban (`response_generation_persona.txt`). Same retrieval (wRRF BM25+metadata+lyrics), same LM (Qwen 2.5-1.5B). **Blind-A only** — no devset validation this round.

**Prediction:** LLM-judge 3.15 → ~3.55 (prior-branch: persona lifts Gemini +0.40). Composite 0.33 → ~0.36. nDCG@20 / CatDiv unchanged (response-only change). LexDiv may tick down slightly (persona compresses vocabulary into specificity — wash in prior-branch data).

**Submission budget warning:** every run here costs one CodaBench slot. Per plan §2.6, verify headroom before submitting.

Output → `/content/prediction.zip` (CodaBench-compliant) + Drive copy tagged with TID.

In [ ]:
# 1) Verify GPU.
!nvidia-smi | head -20

In [ ]:
# 2) FORCE-FRESH clone fresh-model from GitHub.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026-lora-tutorial
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026-lora-tutorial
%cd /content/recsys2026-lora-tutorial

print('\n=== CODE VERSION CHECK ===')
!git log -1 --pretty=format:'commit:  %h%nauthor:  %an%ndate:    %ai%nsubject: %s'
print()
!echo -n 'branch:  ' && git rev-parse --abbrev-ref HEAD

In [ ]:
# 3) Install pinned deps.
!pip install -q -r requirements.txt
!python -c "import torch, transformers, bm25s; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# 4) Experiment parameters (baked in for exp 022 Blind-A).
TID = '022-persona-qwen15b-blindsetA'
BATCH_SIZE = 32
ATTN = 'sdpa'
# import os; os.environ['HF_TOKEN'] = 'hf_...'

In [ ]:
# 5) Run Blind-A inference (80 rows, ~2-3 min on A100).
!cd music-crs-baselines && PYTORCH_ALLOC_CONF=expandable_segments:True \
    python run_inference_blindset.py \
    --tid {TID} \
    --eval_dataset blindset_A \
    --batch_size {BATCH_SIZE} \
    --device cuda \
    --attn_implementation {ATTN}

In [ ]:
# 6) Validate + package CodaBench-compliant prediction.zip (singular prediction.json at root).
import json, os, shutil
SRC = f'music-crs-baselines/exp/inference/blindset_A/{TID}.json'
assert os.path.isfile(SRC), f'inference output missing at {SRC}'
with open(SRC) as f:
    rows = json.load(f)
print(f'rows: {len(rows)} (Blind-A expects 80)')
assert len(rows) == 80
sample = rows[0]
required = {'session_id', 'user_id', 'turn_number', 'predicted_track_ids', 'predicted_response'}
assert not (required - set(sample.keys())), f'missing keys: {required - set(sample.keys())}'
assert len(sample['predicted_track_ids']) == 20
assert sample['predicted_response'].strip()
print(f'sample response[0]: {sample["predicted_response"][:200]!r}')
# Quick prompt-effect sanity: check for the 4 banned words in the first 10 responses.
banned = ['absolutely', 'fantastic', 'truly', 'amazing']
hits = sum(1 for r in rows[:10] for w in banned if w in r['predicted_response'].lower())
print(f'banned-word occurrences in first 10 responses: {hits} (lower is better; stock prompt typically ~4-6)')

stage = '/content/_stage_prediction'
shutil.rmtree(stage, ignore_errors=True); os.makedirs(stage, exist_ok=True)
shutil.copy(SRC, os.path.join(stage, 'prediction.json'))  # singular, at root
!cd {stage} && rm -f /content/prediction.zip && zip -q /content/prediction.zip prediction.json
!unzip -l /content/prediction.zip
print('\nprediction.zip ready — CodaBench-compliant.')

In [ ]:
# 7a) Browser download of prediction.zip.
from google.colab import files
files.download('/content/prediction.zip')

In [ ]:
# 7b) Drive backup tagged with TID so Blind-A ships don't clobber each other.
from google.colab import drive
import os, shutil
drive.mount('/content/drive')
dst = '/content/drive/MyDrive/recsys2026-predictions'
os.makedirs(dst, exist_ok=True)
shutil.copy('/content/prediction.zip', f'{dst}/{TID}__prediction.zip')
shutil.copy(f'music-crs-baselines/exp/inference/blindset_A/{TID}.json', dst)
!ls -lh {dst}

## Ship: upload `prediction.zip` to CodaBench.

After score returns, ping Claude with the 4 numbers (composite, nDCG@20, LexDiv, LLM). Claude will:
- Append a `[blindA]` row to `documents/submissions_log.md`
- Compute Δ vs exp 021 (our prior blind baseline)
- Validate or falsify H-4 (persona lifts LLM +0.40)
- Archive the 80 (query, response, score) triples to `documents/blind_responses_scored.md` for Judge-behavior-notes mining